# Aggregations, Grouping & Window Functions

### Summarizing data efficiently with groupBy and advanced windowing

If you’ve been following along from the beginning, you now know how to load data, explore it, and perform basic transformations. But here’s the truth: in most real-world pipelines, you won’t just be cleaning columns — you’ll be summarizing data.

Business leaders rarely ask, “Can you give me every row of the sales log?” Instead, they ask:

- “What was our total revenue by region this week?”
- “Which store had the highest sales yesterday?”
- “What’s the running total of revenue so far this month?”

All of these are aggregations — ways of grouping data and producing summaries. And in Spark, this is where groupBy and window functions shine.

The Dataset
We’ll keep using the same sales.csv but expand it with more rows to make aggregations meaningful:

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("day9-demo").getOrCreate()

df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/sales/sales1.csv")
df.show()

# Aggregations with groupBy

The simplest way to summarize is to group data by one or more columns and then apply aggregation functions like sum, avg, count, or max.

In [0]:
from pyspark.sql.functions import sum, avg, max, count

# Total revenue per region
df.groupBy("region").agg(sum("revenue").alias("total_revenue")).show()

In [0]:
df.groupBy("region", "date").agg(sum("revenue").alias("daily_revenue")).show()

Built-in Aggregations
Spark gives us many aggregators out of the box:

- sum
- avg
- max
- min
- count
- approx_count_distinct

For example, finding the average revenue per store:

# Average

In [0]:
# Total revenue per region
df.groupBy("region").agg(avg("revenue").alias("avg_revenue")).show()

# Max

In [0]:
from pyspark.sql.functions import max

# Max revenue per region
df.groupBy("region").agg(max("revenue").alias("max_revenue")).show()

# Min

In [0]:
from pyspark.sql.functions import min

# Minimum revenue per region
display(
  df.groupBy("region").agg(
    min("revenue").alias("min_revenue")
  )
)

# Count

In [0]:
from pyspark.sql.functions import count

# Minimum revenue per region
display(
  df.groupBy("region").agg(
    count("revenue").alias("min_revenue")
  )
)

# Running Total per Region

### Window Functions — Beyond Simple Grouping

Grouping collapses rows into summaries. But what if you want running totals, ranks, or comparisons within groups — while still keeping individual rows? That’s where window functions come in.

Think of windows as adding a “lens” over your data. You define a partition (which rows belong together) and an order (how to sort them). Then you apply aggregations within that lens.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

# Define window: partition by region, ordered by date
window_spec = Window.partitionBy("region").orderBy("date")

# Running total revenue
df.withColumn("running_total", sum("revenue").over(window_spec)).show()

Notice how Spark didn’t collapse rows. It preserved each day’s data but added a new column showing the cumulative total per region.

### Example: Ranking Stores by Revenue

In [0]:
from pyspark.sql.functions import rank

# Rank stores by revenue within each region
window_rank = Window.partitionBy("region").orderBy(col("revenue").desc())

df.withColumn("rank", rank().over(window_rank)).show()

Now you can see rankings inside each region. This is incredibly useful for leaderboards, top-N analysis, and performance monitoring.

### A Real-Life Analogy
Imagine you’re a teacher grading students across different classes.

A groupBy would answer: “What is the average score per class?” — you collapse students into one summary per class.

A window function would answer: “What is each student’s rank within their class?” — you keep every student’s row, but add extra context.

Both are aggregations, but windowing lets you see individuals in the context of their group.

### Wrapping Up
Today you learned two of the most powerful tools in Spark:

Aggregations with groupBy: collapse rows to compute totals, averages, counts, and more.

Window functions: enrich rows with running totals, ranks, and comparisons inside partitions.

As a data engineer, you’ll use these constantly. Whether it’s rolling up KPIs for dashboards or ranking transactions for fraud detection, aggregations are your bread and butter.

That’s Day 9. You’ve just learned how to summarize data efficiently with groupBy and window functions.